In [2]:
import pandas as pd
import numpy as np

In [103]:
class Problem:
    def __init__(self, path="data/TSPA.csv"):
        self.df = pd.read_csv(path, sep=";", header=None)
        
        cord = self.df.values[:, :-1]
        dist = np.sqrt(np.sum((cord[None, :, :] - cord[:, None, :]) ** 2, axis=2))
        
        self.dist = np.floor(dist + 0.5).astype(int)
        self.cost = self.df.values[:, -1]

    def get_n(self):
        return len(self.df)

    def eval(self, nodes: np.array, end=True):
        assert nodes.dtype == np.int64
        assert len(set(nodes)) == len(nodes)

        n = self.get_n()
        n_nodes = n // 2
        if end == True and len(nodes) != n_nodes:
            raise ValueError("The final path does not have the required length!")

        pairs = np.column_stack((nodes, np.roll(nodes, -1)))
        if not end:
            pairs = pairs[:-1]
        total_length = np.sum(self.dist[pairs[:, 0], pairs[:, 1]])
        total_cost = np.sum(self.cost[nodes])

        return total_length + total_cost


In [109]:
def random_solution(problem: Problem):
    n = problem.get_n()
    n_nodes = n // 2
    return np.random.choice(n, size=n_nodes, replace=False)

In [108]:
def nearest_neighbor_end(problem: Problem, start=None):
    n = problem.get_n()
    n_nodes = n // 2
    if start is None:
        start = np.random.randint(n)

    path = [start]
    unvisited = np.ones(n, dtype=bool)
    unvisited[start] = False

    while len(path) < n_nodes:
        scores = problem.dist[path[-1]] + problem.cost
        scores = np.where(unvisited, scores, np.inf)
        nxt = int(np.argmin(scores))
        path.append(nxt)
        unvisited[nxt] = False

    return np.array(path, dtype=np.int64)

In [107]:
def nearest_neighbor_any(problem: Problem, start=None):
    n = problem.get_n()
    n_nodes = n // 2
    if start is None:
        start = np.random.randint(n)

    d = problem.dist
    path = [start]
    visited = {start}

    while len(path) < n_nodes:
        best_delta, best_node, best_pos = None, None, None

        for j in range(n):
            if j in visited:
                continue

            for i in range(len(path) + 1):
                if i == 0:
                    delta = d[j, path[0]]
                elif i == len(path):
                    delta = d[path[-1], j]
                else:
                    a, b = path[i - 1], path[i]
                    delta = d[a, j] + d[j, b] - d[a, b]
                delta += problem.cost[j]

                if best_delta is None or delta < best_delta:
                    best_delta, best_node, best_pos = delta, j, i

        path.insert(best_pos, best_node)
        visited.add(best_node)

    return np.array(path, dtype=np.int64)

In [116]:
def eval(f: callable, p: Problem, iter=200):
    values = []
    best = None
    best_val = float("inf")

    for _ in range(iter):
        solution = f(p)
        val = p.eval(solution)
        values.append(val)
        if val < best_val:
            best = solution
            best_val = val
    
    values = np.array(values)

    print(f"Min: {np.min(values)}")
    print(f"Max: {np.max(values)}")
    print(f"Mean: {np.mean(values)}")

    return best

In [135]:
p = Problem("data/TSPB.csv")

In [136]:
s = eval(nearest_neighbor_end, p)
df = pd.DataFrame({"solution": s})
df.to_parquet("data/nearest_neighbor_end_tspb")

Min: 56007
Max: 63166
Mean: 59553.275
